#  IS PYDANTIC ?

## 1 INSPECT THE CONSTRUCTOR

### Outcome A — is_pydantic=False (plain Python class):

In [1]:
import inspect
import typing

from langchain_core.output_parsers import JsonOutputParser
from langchain_core.prompts.few_shot_with_templates import FewShotPromptWithTemplates
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pydantic import BaseModel


In [2]:
is_pydantic = BaseModel in RecursiveCharacterTextSplitter.__mro__
print("Pydantic-based:", is_pydantic)
inspect.signature(RecursiveCharacterTextSplitter.__init__)

Pydantic-based: False


<Signature (self, separators: 'Optional[list[str]]' = None, keep_separator: "Union[bool, Literal['start', 'end']]" = True, is_separator_regex: 'bool' = False, **kwargs: 'Any') -> 'None'>


> The MRO already told you what to expect: no BaseModel means a hand-written `__init__`, so the signature is the real, complete API — every knob with its default, right there. 
>> Read what each param does and start using them directly. You're done — do not go to Step 2, model_fields doesn't exist on this class and will raise `AttributeError`.

### Outcome B — is_pydantic=True (Pydantic model):

In [3]:
inspect.signature(JsonOutputParser.__init__)
# (self, separators=None, keep_separator=True, is_separator_regex=False, **kwargs) -> None

is_pydantic = BaseModel in JsonOutputParser.__mro__
print("Pydantic-based:", is_pydantic)
inspect.signature(JsonOutputParser.__init__)

Pydantic-based: True


<Signature (self, *args: Any, **kwargs: Any) -> None>

> This means the class has no custom `__init__` — it inherits a generic one from Pydantic's BaseModel, which just accepts kwargs and assigns them to matching fields, so the signature carries no per-field information. 
>> Go straight to Step 2, where model_fields is guaranteed to exist and holds the real API surface.

## Step 2 — List fields, rank by two signals
> Reflect: filter to just the rows where nullable_type=True — those are your candidates for "this field probably gates an if/else in the source." 
>> name and pydantic_object qualifY as None is a semantically distinct, code-branching value for that field, not just "you may omit this". 
>>>diff doesn't qualify , even though all three are optional-to-pass. On diff, its `default=False` is just a normal off-switch and `required=False` with a non-nullable type are just plain on/off flags, not schema-shaped branch points. Go check json.py again and you'll see diff is only ever tested truthy/falsy, never is None.

In [4]:
for name, info in JsonOutputParser.model_fields.items():
    args = typing.get_args(info.annotation)
    nullable = type(None) in args
    print(f"{name:20} required={info.is_required()!s:6} nullable={nullable!s:6} type={info.annotation}")


name                 required=False  nullable=True   type=typing.Optional[str]
diff                 required=False  nullable=False  type=<class 'bool'>
pydantic_object      required=False  nullable=True   type=typing.Optional[type[~TBaseModel]]


In [5]:
# FewShotPromptWithTemplates has base model, so it is Pydantic.
for name, info in FewShotPromptWithTemplates.model_fields.items():
    args = typing.get_args(info.annotation)
    nullable = type(None) in args
    print(f"{name:20} required={info.is_required()!s:6} nullable={nullable!s:6} type={info.annotation}")

name                 required=False  nullable=True   type=typing.Optional[str]
input_variables      required=True   nullable=False  type=list[str]
optional_variables   required=False  nullable=False  type=list[str]
input_types          required=False  nullable=False  type=typing.Dict[str, typing.Any]
output_parser        required=False  nullable=True   type=typing.Optional[langchain_core.output_parsers.base.BaseOutputParser]
partial_variables    required=False  nullable=False  type=collections.abc.Mapping[str, typing.Any]
metadata             required=False  nullable=True   type=typing.Optional[typing.Dict[str, typing.Any]]
tags                 required=False  nullable=True   type=typing.Optional[list[str]]
examples             required=False  nullable=True   type=typing.Optional[list[dict]]
example_selector     required=False  nullable=False  type=typing.Any
example_prompt       required=True   nullable=False  type=<class 'langchain_core.prompts.prompt.PromptTemplate'>
suffix         

### Outcome A — nullable=True, or type is Any/suggestively named:

> `pydantic_object  required=False  nullable=True  type=typing.Optional[type]` <br>
`example_selector required=False  nullable=False type=typing.Any`
>> Both rows are high priority, for different reasons. `nullable=True` means `None` is a literal member of the type's `Union`, so it's a strong candidate for a real `if self.field is None:` branch in the source. 
>>> `Any` or a name like `*_object/*_schema/*_selector` slips past the nullable check but often hides the same kind of branch — sometimes because of a circular-import workaround, sometimes genuinely accepting arbitrary data. Either way, this field goes to Step 3.

### Outcome B — required=False, nullable=False, plain type like bool or str:




> `diff  required=False  nullable=False  type=<class 'bool'>`
>> This is just an on/off flag with a default — omitting it doesn't unlock different code paths, it just picks the default value of a single behavior. 
>>> Lower priority for Step 3, though not zero: quickly confirm in the source that it's only ever tested truthy `if self.diff:`, not `is None`. If it turns out to be tested against a sentinel instead of `None`, treat it like Outcome A.

## Step 3 — Grep the field in source to check for branching

> Steps 1 and 2 only narrow down candidates worth checking — being Pydantic-based (Step 1) and having a nullable/`Any`/suggestively-named field (Step 2) are both necessary conditions to bother looking, never sufficient proof that a branch exists.
>> Step 3 is the only step that reads actual behavior instead of type shape, so it's the only one that can confirm or rule out branch-splitting.

In [6]:
print(inspect.getsourcefile(JsonOutputParser))

# grep the field across the whole class first, to see every place it's touched
src = inspect.getsource(JsonOutputParser)
for lineno, line in enumerate(src.splitlines(), 1):
    if "pydantic_object" in line:
        print(lineno, line)

# then pull just the method where the branch actually lives
print(inspect.getsource(JsonOutputParser.get_format_instructions))

/Users/marcelohanones/Developer/personal/IBM-RAG-and-Agentic-AI/.venv/lib/python3.12/site-packages/langchain_core/output_parsers/json.py
11     pydantic_object: Annotated[Optional[type[TBaseModel]], SkipValidation()] = None  # type: ignore[valid-type]
20     def _get_schema(pydantic_object: type[TBaseModel]) -> dict[str, Any]:
21         if issubclass(pydantic_object, pydantic.BaseModel):
22             return pydantic_object.model_json_schema()
23         return pydantic_object.schema()
74         if self.pydantic_object is None:
77         schema = dict(self._get_schema(self.pydantic_object).items())
    def get_format_instructions(self) -> str:
        """Return the format instructions for the JSON output.

        Returns:
            The format instructions for the JSON output.
        """
        if self.pydantic_object is None:
            return "Return a JSON object."
        # Copy schema to avoid altering original Pydantic schema.
        schema = dict(self._get_schema(self.

In [7]:
print(inspect.getsourcefile(FewShotPromptWithTemplates))

# grep both fields across the whole class first, to see every place they're touched
src = inspect.getsource(FewShotPromptWithTemplates)
for lineno, line in enumerate(src.splitlines(), 1):
    if "example_selector" in line or "examples" in line:
        print(lineno, line)

# then pull just the method where the branch actually lives
print(inspect.getsource(FewShotPromptWithTemplates._get_examples))

/Users/marcelohanones/Developer/personal/IBM-RAG-and-Agentic-AI/.venv/lib/python3.12/site-packages/langchain_core/prompts/few_shot_with_templates.py
2     """Prompt template that contains few shot examples."""
4     examples: Optional[list[dict]] = None
6     Either this or example_selector should be provided."""
8     example_selector: Any = None
9     """ExampleSelector to choose the examples to format into the prompt.
10     Either this or examples should be provided."""
16     """A PromptTemplate to put after the examples."""
19     """String separator used to join the prefix, the examples, and suffix."""
22     """A PromptTemplate to put before the examples."""
42     def check_examples_and_selector(cls, values: dict) -> Any:
43         """Check that one and only one of examples/example_selector are provided."""
44         examples = values.get("examples")
45         example_selector = values.get("example_selector")
46         if examples and example_selector:
47             msg =

### Outcome A — you find if self.field is None: ... else: ... (or is SENTINEL):

`JsonOutputParser` branches on `pydantic_object`: <br>
`if self.pydantic_object is None:` <br>
&nbsp;&nbsp;&nbsp;&nbsp;`return "Return a JSON object."` <br>
`else: build schema-driven instructions`

`FewShotPromptWithTemplates` branches the same way, just split across **two** fields instead of one: <br>
`if self.examples is not None: return self.examples` <br>
`if self.example_selector is not None: return self.example_selector.select_examples(kwargs)`

> Both confirm the field(s) really are a mode-switch — you've found the two (or more) ways of doing the same thing that started this whole investigation.
>> Note both branches and what each produces; that's the answer you came here for. If it's a sentinel rather than `None`, flag it mentally — Step 2's nullable check would have missed this one entirely, so don't trust that check alone next time on a similarly-shaped field.

### Outcome B — only truthy checks (if self.field:), or the field never appears beyond assignment:

`JsonOutputParser`'s `diff` field does **not** branch: <br>
`if self.diff:` <br>
&nbsp;&nbsp;&nbsp;&nbsp;`yield JsonPatch(...)`

> This confirms `diff` is a plain flag, not a schema-shaped branch — matches what Step 2 predicted for Outcome B fields. This is a per-field verdict, not a per-class one: the same `JsonOutputParser` that branches on `pydantic_object` (Outcome A) also carries this non-branching field — one class can hold both kinds at once.
>> If a field doesn't appear anywhere in the method bodies at all, it's inert for behavior (kept for validation, serialization, or backward compatibility) and safe to deprioritize. Either way, you're done investigating this field — move to the next candidate from Step 2's list.

In [21]:
from langchain_core.documents import Document

# Step 1
print(">>>> is_pydantic:", BaseModel in Document.__mro__)

# Step 2
for name, info in Document.model_fields.items():
    args = typing.get_args(info.annotation)
    nullable = type(None) in args
    print(f"{name:12} required={info.is_required()!s:6} nullable={nullable!s:6} type={info.annotation}")

# Step 3
src = inspect.getsource(Document)
branch_found = any("is None" in line or "SENTINEL" in line for line in src.splitlines())
branch_found_status = "Yes" if branch_found else "No"

print(f""" \n>>>> branch found in source: {branch_found_status}""")
# print(src)

>>>> is_pydantic: True
id           required=False  nullable=True   type=typing.Optional[str]
metadata     required=False  nullable=False  type=<class 'dict'>
page_content required=True   nullable=False  type=<class 'str'>
type         required=False  nullable=False  type=typing.Literal['Document']
 
>>>> branch found in source: No


### Outcome C — no branch exists anywhere in the class, despite Steps 1 and 2 both checking out:

`Document` does **not** branch anywhere, despite passing Steps 1 and 2: `BaseModel in MRO` → `True` (Step 1 ✅). `page_content: str` (required) and `type: Literal["Document"] = "Document"` — neither nullable, `Any`, nor suggestively named, so Step 2 doesn't even flag a strong candidate. Grepping across every method turns up no `is None`, no sentinel, no alternate code path.
> This is the case that closes the loop: a class can be Pydantic (Step 1 ✅) and even have optional-ish fields worth a glance (Step 2 flags candidates or comes up empty), yet Step 3 still turns up nothing — every method just reads/formats the stored data, no field gates a different behavior.
>> The verdict here is **not branch-splitting**, just a validated data container. Only Step 3 coming back empty across every flagged field justifies that conclusion; Steps 1–2 alone never do.

## Try it on a class you haven't seen yet, to check the method generalizes

> Run that and see if any field jumps out as an Optional[...] = None mode-switch — then chase it into the source the same way. That's the real test of whether the method stuck, versus whether you just followed along on JsonOutputParser.

In [9]:
from langchain_core.prompts import PromptTemplate

# step 1
print(inspect.signature(PromptTemplate.__init__))          

(self, *args: Any, **kwargs: Any) -> None


In [10]:
# Step 2
for name, info in PromptTemplate.model_fields.items():
    args = typing.get_args(info.annotation)
    nullable_type = type(None) in args          # None is a real, distinct value for this field
    print(f"{name:19} required={info.is_required()!s:6} "
          f"nullable_type={nullable_type!s:6} default={info.default!r}")

name                required=False  nullable_type=True   default=None
input_variables     required=True   nullable_type=False  default=PydanticUndefined
optional_variables  required=False  nullable_type=False  default=[]
input_types         required=False  nullable_type=False  default=PydanticUndefined
output_parser       required=False  nullable_type=True   default=None
partial_variables   required=False  nullable_type=False  default=PydanticUndefined
metadata            required=False  nullable_type=True   default=None
tags                required=False  nullable_type=True   default=None
template            required=True   nullable_type=False  default=PydanticUndefined
template_format     required=False  nullable_type=False  default='f-string'
validate_template   required=False  nullable_type=False  default=False


In [11]:
# step 3
print(inspect.getdoc(PromptTemplate))                        

Prompt template for a language model.

A prompt template consists of a string template. It accepts a set of parameters
from the user that can be used to generate a prompt for a language model.

The template can be formatted using either f-strings (default), jinja2,
or mustache syntax.

*Security warning*:
    Prefer using `template_format="f-string"` instead of
    `template_format="jinja2"`, or make sure to NEVER accept jinja2 templates
    from untrusted sources as they may lead to arbitrary Python code execution.

    As of LangChain 0.0.329, Jinja2 templates will be rendered using
    Jinja2's SandboxedEnvironment by default. This sand-boxing should
    be treated as a best-effort approach rather than a guarantee of security,
    as it is an opt-out rather than opt-in approach.

    Despite the sand-boxing, we recommend to never use jinja2 templates
    from untrusted sources.

Example:

    .. code-block:: python

        from langchain_core.prompts import PromptTemplate

        

In [12]:
print(inspect.getsourcefile(PromptTemplate))

/Users/marcelohanones/Developer/personal/IBM-RAG-and-Agentic-AI/.venv/lib/python3.12/site-packages/langchain_core/prompts/prompt.py


# End